# FP-Cox Optimizer — SaDE v12 (Review Fixes Applied)

Self-Adaptive Differential Evolution for Fractional Polynomial Cox model selection.

**Objective:** Pure BIC minimisation (no penalizer, as requested).

---

## Fixes applied compared to v11

### Tier 1 (correctness)
1. **BIC uses `log(n_total)` everywhere** — was `log(n_events)` for Cox and `log(n_total)` for Weibull in v11, so ΔBIC comparisons across models were meaningless. Now all five models use the same `n_total` convention (matches `mfp2`).
2. **`_repair` uses modular wrap** instead of reflection + clip — reflection pushed out-of-bounds trials back out of bounds, and the final `np.clip` then pinned them to the edge, biasing the search toward boundary powers.
3. **Canonical SaDE spec**: `F ~ Normal(0.5, 0.3)` (was Cauchy — JADE-style) and `CRm` is updated to the **median** of successful CRs (was mean). These are the Qin–Suganthan 2009 defaults.
4. **FP terms are centred** on their sample mean before Cox fitting — this removes near-singularity between `x^p` and `x^p·ln(x)` for repeated powers and improves numerical stability.
5. **`_canonical_key` is rewritten to key on powers, not indices** — the v11 version mapped `(None, p)` and `(p, None)` to different cache keys, so we were paying double for half the hits.

### Tier 2 (algorithm design)
6. **Data-adaptive population sizing** (Storn-Price 1997 + event-count cap):
   - Baseline `NP = pop_multiplier * D` with `pop_multiplier=10` (widely-used SaDE/JADE/jDE default).
   - Floor `max(20, 4*D)` — enough diversity for DE/rand/2 and DE/curr-to-best/2.
   - Cap `min(15*D, n_events)` — don't exceed event count; the Cox partial-likelihood surface has limited effective resolution when events are sparse, so larger populations waste evaluations.
   - v11 used a fixed `popsize=12*D` (later bumped to `20*D`) with no data awareness. PBC (10 cov, 161 events, D=20) now uses NP=161 instead of 400 — roughly 2.4x fewer Cox fits per generation with equivalent coverage.
   - `maxiter=60, patience=maxiter//2` retained from v12-initial.
7. **MFP ranks covariates by Wald p-value** before cycle 1 (Royston–Sauerbrei spec) — v11 iterated in user-provided order.
8. **MFP feasibility guard** — if all FP fits are numerically infeasible, the variable is marked dropped instead of returning `(1, 1)` with infinite deviance.
9. **MFP sensitivity check on extended power grid** (same 15 powers as SaDE, minus `None`) — lets us see whether SaDE's advantage comes from the optimiser or just from having a bigger menu.

### Tier 3 (simulation study)
10. **Out-of-sample C-index** is computed on the 10% holdout for all four models in each of the 1000 simulations. v11 only collected in-sample (training) C-index, which is upward-biased by model complexity.

### Removed (time wasters per user request)
- `run_cross_validation` — redundant with the simulation study.
- `run_sklearn_cv` — same.
- `validate_algorithm` — CV<1% from 10 seeds does not prove convergence; only stability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, WeibullAFTFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
import warnings
from collections import deque
from dataclasses import dataclass, field
from typing import List, Union, Optional
from scipy import stats as scipy_stats
from itertools import combinations, combinations_with_replacement

warnings.filterwarnings('ignore')

try:
    from sksurv.metrics import integrated_brier_score
    from sksurv.util import Surv
    HAS_SKSURV = True
except ImportError:
    HAS_SKSURV = False
    print('scikit-survival not found — IBS skipped.')


In [ ]:
# Result container for SaDE
@dataclass
class _SaDEResult:
    x:       np.ndarray
    fun:     float
    nfev:    int
    ngen:    int
    history: List[float] = field(default_factory=list)


In [ ]:
# _choose: picks k distinct indices excluding those in `excl`
def _choose(n: int, k: int, excl: Union[int, list], rng) -> np.ndarray:
    mask = np.ones(n, dtype=bool)
    if isinstance(excl, int):
        excl = [excl]
    for e in excl:
        mask[e] = False
    pool = np.where(mask)[0]
    if len(pool) < k:
        return rng.choice(pool, size=k, replace=True)
    return rng.choice(pool, size=k, replace=False)


In [ ]:
# Mutation strategies — standard SaDE pool

def _s1(pop, F, t, b, rng):   # DE/rand/1
    r1, r2, r3 = _choose(len(pop), 3, t, rng)
    return pop[r1] + F * (pop[r2] - pop[r3])

def _s2(pop, F, t, b, rng):   # DE/current-to-best/2
    r1, r2, r3, r4 = _choose(len(pop), 4, [t, b], rng)
    return (pop[t] + F*(pop[b]-pop[t]) + F*(pop[r1]-pop[r2]) + F*(pop[r3]-pop[r4]))

def _s3(pop, F, t, b, rng):   # DE/current-to-rand/1
    r1, r2, r3 = _choose(len(pop), 3, t, rng)
    return pop[t] + F*(pop[r1]-pop[t]) + F*(pop[r2]-pop[r3])

def _s4(pop, F, t, b, rng):   # DE/rand/2
    r1, r2, r3, r4, r5 = _choose(len(pop), 5, t, rng)
    return pop[r1] + F*(pop[r2]-pop[r3]) + F*(pop[r4]-pop[r5])

_STRATS = [_s1, _s2, _s3, _s4]
_NS     = len(_STRATS)


In [ ]:
# ---------------------------------------------------------------------------
# Boundary repair: modular wrap (review fix #2)
# ---------------------------------------------------------------------------
# v11 used reflection + clip, which biased trials toward boundary indices
# because reflecting a far-out value produces another far-out value and
# the subsequent np.clip pinned it to the edge. Modular wrap preserves the
# uniform distribution over feasible integers.
# ---------------------------------------------------------------------------

def _repair(v, lb, ub):
    v    = np.round(v).astype(int)
    span = (ub - lb + 1)
    return lb + np.mod(v - lb, span)

def _cross(x, v, CR, rng):
    d = len(x)
    m = rng.random(d) < CR
    m[rng.integers(d)] = True
    return np.where(m, v, x)


In [ ]:
# ---------------------------------------------------------------------------
# SaDE engine — canonical Qin–Suganthan 2009 spec (review fix #3)
# ---------------------------------------------------------------------------
#   F  ~ Normal(0.5, 0.3), clipped to (0, 2]        [original SaDE]
#   CR ~ Normal(CRmk, 0.1), clipped to [0, 1]       [original SaDE]
#   CRmk updated as the MEDIAN of successful CRs    [original SaDE]
#   Strategy probabilities adapted from success/failure counts over LP gens.
# ---------------------------------------------------------------------------

class _SaDE:
    """
    Self-Adaptive Differential Evolution for integer search spaces.

    Strategy pool: DE/rand/1, DE/current-to-best/2,
                   DE/current-to-rand/1, DE/rand/2.
    """

    def __init__(self, func, bounds, pop_size=50, max_evals=1000,
                 lp=10, patience=10, seed=None, callback=None):
        self.func     = func
        self.dim      = len(bounds)
        self.lb       = np.array([b[0] for b in bounds], dtype=int)
        self.ub       = np.array([b[1] for b in bounds], dtype=int)
        self.N        = pop_size
        self.maxev    = max_evals
        self.lp       = lp
        self.patience = patience
        self.cb       = callback
        self.rng      = np.random.default_rng(seed)
        self.p        = np.ones(_NS) / _NS
        self.crm      = np.full(_NS, 0.5)
        self.ns       = [deque() for _ in range(_NS)]
        self.nf       = [deque() for _ in range(_NS)]
        self.crok     = [deque() for _ in range(_NS)]
        self.strategy_counts  = np.zeros(_NS, dtype=int)
        self.strategy_success = np.zeros(_NS, dtype=int)

    def run_opt(self, init_pop=None):
        if init_pop is not None and init_pop.shape == (self.N, self.dim):
            pop = np.clip(np.round(init_pop).astype(int), self.lb, self.ub)
        else:
            pop = self.rng.integers(self.lb, self.ub + 1, size=(self.N, self.dim))

        fit        = np.array([self.func(pop[i]) for i in range(self.N)])
        nfev       = self.N
        bi         = int(np.argmin(fit))
        hist       = [float(fit[bi])]
        gen        = 0
        no_improve = 0
        best_ever  = float(fit[bi])

        while nfev < self.maxev:
            gen += 1
            for i in range(self.N):
                if nfev >= self.maxev:
                    break
                k  = self.rng.choice(_NS, p=self.p)
                # Canonical SaDE: F ~ Normal(0.5, 0.3), clipped to (0, 2]
                F  = float(np.clip(self.rng.normal(0.5, 0.3), 1e-6, 2.0))
                CR = float(np.clip(self.rng.normal(self.crm[k], 0.1), 0.0, 1.0))
                v  = _STRATS[k](pop, F, i, bi, self.rng).astype(float)
                u  = _repair(_cross(pop[i].astype(float), v, CR, self.rng),
                             self.lb, self.ub)
                fu = self.func(u);  nfev += 1
                self.strategy_counts[k] += 1
                if fu <= fit[i]:
                    pop[i], fit[i] = u, fu
                    self.ns[k].append(1)
                    self.crok[k].append(CR)
                    self.strategy_success[k] += 1
                    if fu < fit[bi]: bi = i
                else:
                    self.nf[k].append(1)

            if gen % self.lp == 0:
                self._upd_p()
                self._upd_crm()

            hist.append(float(fit[bi]))
            if self.cb: self.cb(pop[bi], fit[bi], gen)

            if self.patience > 0:
                if fit[bi] < best_ever - 1e-8:
                    best_ever = float(fit[bi]);  no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= self.patience:
                    print(f'  Early stop at gen {gen} '
                          f'(no improvement for {self.patience} gens, evals: {nfev})')
                    break

        return _SaDEResult(x=pop[bi].copy(), fun=float(fit[bi]),
                           nfev=nfev, ngen=gen, history=hist)

    def _upd_p(self):
        ns = np.array([sum(q) for q in self.ns], dtype=float)
        nf = np.array([sum(q) for q in self.nf], dtype=float)
        with np.errstate(divide='ignore', invalid='ignore'):
            r = np.where(ns+nf > 0, ns/(ns+nf), 0.0)
        tot = r.sum()
        if tot > 0:
            self.p = 0.05 + 0.95*r/tot
            self.p /= self.p.sum()
        for k in range(_NS):
            while len(self.ns[k]) > self.lp: self.ns[k].popleft()
            while len(self.nf[k]) > self.lp: self.nf[k].popleft()

    def _upd_crm(self):
        # Canonical SaDE: update CRm as the MEDIAN of successful CRs
        for k in range(_NS):
            if self.crok[k]:
                self.crm[k] = float(np.median(list(self.crok[k])))
            while len(self.crok[k]) > self.lp: self.crok[k].popleft()


print('SaDE engine loaded (canonical Qin–Suganthan spec).')


In [ ]:
# =========================================================================
# Traditional MFP (Multivariable Fractional Polynomials) — review fixes
# =========================================================================
# Fix #7: covariates ranked by Wald p-value before cycle 1 (Royston 2008).
# Fix #8: feasibility guard — if all FP fits fail, the variable is dropped.
# Fix #9: accepts a power_set argument so we can run it on the SAME extended
#         grid as SaDE as a fair-comparison sensitivity check.
# =========================================================================

class MFPSelector:
    """
    Traditional Multivariable Fractional Polynomial selector for Cox PH.

    Parameters
    ----------
    alpha_select, alpha_function : float
        Significance levels for variable / function selection (default 0.05).
    max_cycles : int
        Maximum number of MFP cycles (default 10).
    power_set : list
        Candidate powers (default: standard 8-power Royston set).
    """

    STANDARD_POWERS = [-2, -1, -0.5, 0, 0.5, 1, 2, 3]

    def __init__(self, alpha_select=0.05, alpha_function=0.05, max_cycles=10,
                 power_set=None):
        self.alpha_select   = alpha_select
        self.alpha_function = alpha_function
        self.max_cycles     = max_cycles
        self.power_set      = power_set or self.STANDARD_POWERS

    def _build_fp_terms(self, x, p1, p2=None):
        log_x = np.log(x)
        z1 = log_x if p1 == 0 else np.power(x, p1)
        if p2 is None:
            return {'fp1': z1}
        if p2 == p1:
            z2 = z1 * log_x          # repeated power
        elif p2 == 0:
            z2 = log_x
        else:
            z2 = np.power(x, p2)
        return {'fp1': z1, 'fp2': z2}

    def _fit_cox_deviance(self, df, duration_col, event_col, strata=None):
        try:
            cph = CoxPHFitter(penalizer=0.0)
            cph.fit(df, duration_col=duration_col, event_col=event_col,
                    strata=strata, show_progress=False)
            return -2 * cph.log_likelihood_, len(cph.params_), cph
        except Exception:
            return 1e15, 0, None

    def _best_fp1(self, base_df, col_x, duration_col, event_col, strata=None):
        x_vals = base_df[col_x].values
        best_dev, best_p = np.inf, 1
        any_feasible = False
        for p in self.power_set:
            terms = self._build_fp_terms(x_vals, p)
            if not all(np.isfinite(v).all() for v in terms.values()):
                continue
            df_tmp = base_df.copy()
            drop_cols = [c for c in df_tmp.columns if c.startswith(f'_mfp_{col_x}')]
            df_tmp = df_tmp.drop(columns=drop_cols, errors='ignore')
            if col_x in df_tmp.columns:
                df_tmp = df_tmp.drop(columns=[col_x])
            df_tmp[f'_mfp_{col_x}_1'] = terms['fp1']
            dev, _, _ = self._fit_cox_deviance(
                df_tmp, duration_col, event_col, strata)
            if np.isfinite(dev) and dev < 1e14:
                any_feasible = True
            if dev < best_dev:
                best_dev, best_p = dev, p
        return best_p, best_dev, any_feasible

    def _best_fp2(self, base_df, col_x, duration_col, event_col, strata=None):
        x_vals = base_df[col_x].values
        best_dev, best_pp = np.inf, (1, 1)
        any_feasible = False
        for p1, p2 in combinations_with_replacement(self.power_set, 2):
            terms = self._build_fp_terms(x_vals, p1, p2)
            if not all(np.isfinite(v).all() for v in terms.values()):
                continue
            df_tmp = base_df.copy()
            drop_cols = [c for c in df_tmp.columns if c.startswith(f'_mfp_{col_x}')]
            df_tmp = df_tmp.drop(columns=drop_cols, errors='ignore')
            if col_x in df_tmp.columns:
                df_tmp = df_tmp.drop(columns=[col_x])
            df_tmp[f'_mfp_{col_x}_1'] = terms['fp1']
            df_tmp[f'_mfp_{col_x}_2'] = terms['fp2']
            dev, _, _ = self._fit_cox_deviance(
                df_tmp, duration_col, event_col, strata)
            if np.isfinite(dev) and dev < 1e14:
                any_feasible = True
            if dev < best_dev:
                best_dev, best_pp = dev, (p1, p2)
        return best_pp, best_dev, any_feasible

    def _fsp_for_variable(self, base_df, col_x, duration_col, event_col,
                          strata=None):
        """Function Selection Procedure for one continuous variable."""
        from scipy.stats import chi2

        x_vals = base_df[col_x].values

        # Null model (without this variable)
        df_null = base_df.copy()
        drop_cols = [c for c in df_null.columns if c.startswith(f'_mfp_{col_x}')]
        df_null = df_null.drop(columns=drop_cols, errors='ignore')
        if col_x in df_null.columns:
            non_special = [c for c in df_null.columns
                           if c not in [duration_col, event_col]
                           and not (strata and c in strata)]
            if col_x in non_special:
                df_null = df_null.drop(columns=[col_x], errors='ignore')
        dev_null, _, _ = self._fit_cox_deviance(
            df_null, duration_col, event_col, strata)

        # Linear model
        df_lin = base_df.copy()
        drop_cols = [c for c in df_lin.columns if c.startswith(f'_mfp_{col_x}')]
        df_lin = df_lin.drop(columns=drop_cols, errors='ignore')
        if col_x not in df_lin.columns:
            df_lin[col_x] = x_vals
        dev_lin, _, _ = self._fit_cox_deviance(
            df_lin, duration_col, event_col, strata)

        # Best FP1 / FP2
        p1_best, dev_fp1, ok_fp1 = self._best_fp1(
            base_df, col_x, duration_col, event_col, strata)
        pp_best, dev_fp2, ok_fp2 = self._best_fp2(
            base_df, col_x, duration_col, event_col, strata)

        # Feasibility guard (fix #8): if no FP fit worked, drop the variable
        if not ok_fp1 and not ok_fp2:
            return {'selected': False, 'powers': (None, None),
                    'fp_type': 'dropped', 'deviances': {
                        'null': dev_null, 'linear': dev_lin,
                        'fp1': dev_fp1, 'fp2': dev_fp2},
                    'p_values': {}}

        # Step 1: FP2 vs Null (4 df)
        lrt_1 = dev_null - dev_fp2
        p_val_1 = 1 - chi2.cdf(max(0, lrt_1), df=4)
        if p_val_1 > self.alpha_select:
            return {'selected': False, 'powers': (None, None),
                    'fp_type': 'dropped', 'deviances': {
                        'null': dev_null, 'linear': dev_lin,
                        'fp1': dev_fp1, 'fp2': dev_fp2},
                    'p_values': {'fp2_vs_null': p_val_1}}

        # Step 2: FP2 vs Linear (3 df)
        lrt_2 = dev_lin - dev_fp2
        p_val_2 = 1 - chi2.cdf(max(0, lrt_2), df=3)
        if p_val_2 > self.alpha_function:
            return {'selected': True, 'powers': (1, None),
                    'fp_type': 'linear', 'deviances': {
                        'null': dev_null, 'linear': dev_lin,
                        'fp1': dev_fp1, 'fp2': dev_fp2},
                    'p_values': {'fp2_vs_null': p_val_1,
                                 'fp2_vs_lin': p_val_2}}

        # Step 3: FP2 vs FP1 (2 df)
        lrt_3 = dev_fp1 - dev_fp2
        p_val_3 = 1 - chi2.cdf(max(0, lrt_3), df=2)
        if p_val_3 > self.alpha_function:
            return {'selected': True, 'powers': (p1_best, None),
                    'fp_type': 'FP1', 'deviances': {
                        'null': dev_null, 'linear': dev_lin,
                        'fp1': dev_fp1, 'fp2': dev_fp2},
                    'p_values': {'fp2_vs_null': p_val_1,
                                 'fp2_vs_lin': p_val_2,
                                 'fp2_vs_fp1': p_val_3}}

        return {'selected': True, 'powers': pp_best,
                'fp_type': 'FP2', 'deviances': {
                    'null': dev_null, 'linear': dev_lin,
                    'fp1': dev_fp1, 'fp2': dev_fp2},
                'p_values': {'fp2_vs_null': p_val_1,
                             'fp2_vs_lin': p_val_2,
                             'fp2_vs_fp1': p_val_3}}

    def _rank_by_wald(self, df, covariates, duration_col, event_col, strata):
        """
        Fix #7: rank covariates by Wald-test p-value (most significant first).
        Royston & Sauerbrei 2008 §6.1 specify this as the MFP cycle-1 ordering.
        """
        cols = list(covariates) + [duration_col, event_col]
        if strata:
            cols = cols + [c for c in strata if c not in cols]
        df_full = df[cols].copy()
        try:
            cph = CoxPHFitter(penalizer=0.0)
            cph.fit(df_full, duration_col=duration_col, event_col=event_col,
                    strata=strata or None, show_progress=False)
            pvals = cph.summary['p']
            ranked = [c for c in sorted(covariates, key=lambda c: pvals.get(c, 1.0))]
            return ranked
        except Exception:
            return list(covariates)

    def fit(self, df, covariates, duration_col, event_col, strata_cols=None):
        strata = strata_cols or None
        non_cov_cols = [duration_col, event_col]
        if strata:
            non_cov_cols += list(strata)

        # Fix #7: rank covariates by Wald p-value before the first cycle
        covariates_ranked = self._rank_by_wald(
            df, covariates, duration_col, event_col, strata)

        current_powers   = {c: (1, None) for c in covariates_ranked}
        current_selected = {c: True for c in covariates_ranked}
        fsp_info         = {}

        for cycle in range(1, self.max_cycles + 1):
            prev_powers   = dict(current_powers)
            prev_selected = dict(current_selected)

            for cov in covariates_ranked:
                base_df = df[non_cov_cols].copy()
                for other_cov in covariates_ranked:
                    if other_cov == cov or not current_selected[other_cov]:
                        continue
                    p1, p2 = current_powers[other_cov]
                    x = df[other_cov].values
                    terms = self._build_fp_terms(x, p1, p2)
                    for tname, tvals in terms.items():
                        base_df[f'_mfp_{other_cov}_{tname}'] = tvals
                base_df[cov] = df[cov].values

                result = self._fsp_for_variable(
                    base_df, cov, duration_col, event_col, strata)
                current_selected[cov] = result['selected']
                current_powers[cov]   = result['powers']
                fsp_info[cov]         = result

            if (current_powers == prev_powers and
                current_selected == prev_selected):
                break

        fp_types = {}
        for cov in covariates_ranked:
            if not current_selected[cov]:
                fp_types[cov] = 'dropped'
            else:
                p1, p2 = current_powers[cov]
                if p2 is None and p1 == 1:  fp_types[cov] = 'linear'
                elif p2 is None:            fp_types[cov] = 'FP1'
                else:                       fp_types[cov] = 'FP2'

        print(f'\n  MFP converged in {cycle} cycle(s)')
        print(f'  α_select={self.alpha_select}, α_function={self.alpha_function}')
        print(f'  Power set |S| = {len(self.power_set)}')
        print(f'  Ranked order: {covariates_ranked}')
        for cov in covariates_ranked:
            sel = current_selected[cov]
            p   = current_powers[cov]
            ft  = fp_types[cov]
            info = fsp_info.get(cov, {})
            pvals = info.get('p_values', {})
            pv_str = '  '.join(f'{k}={v:.4f}' for k,v in pvals.items())
            print(f'    {cov:<22}: {ft:<8}  powers={p}  [{pv_str}]')

        return {
            'powers':   current_powers,
            'selected': current_selected,
            'fp_types': fp_types,
            'fsp_info': fsp_info,
            'n_cycles': cycle,
        }

    def generate_fp_features(self, df, covariates, mfp_result):
        """Generate FP-transformed features. Centring is applied by the caller."""
        transformed = {}
        for cov in covariates:
            if not mfp_result['selected'][cov]:
                continue
            p1, p2 = mfp_result['powers'][cov]
            x = df[cov].values.astype(float)
            terms = self._build_fp_terms(x, p1, p2)
            if p2 is None:
                transformed[f'{cov}_mfp_{p1}'] = terms['fp1']
            else:
                transformed[f'{cov}_mfp1_{p1}'] = terms['fp1']
                transformed[f'{cov}_mfp2_{p2}'] = terms['fp2']
        return transformed


print('MFPSelector loaded (with Wald-ranked ordering and feasibility guard).')


In [ ]:
# =========================================================================
# FPCoxOptimizer v12 — pure BIC, no penalizer, all review fixes applied
# =========================================================================

class FPCoxOptimizer:
    """
    Models fitted and compared
    --------------------------
    1. Kaplan-Meier (KM) — Non-parametric marginal baseline
    2. Traditional Cox PH — Linear covariates
    3. Weibull AFT — Fully parametric, linear covariates
    4. MFP Cox (traditional FP, 8 powers) — Powers selected by FSP/LRT
    5. FP Cox (SaDE, 15 powers + None) — Powers selected by SaDE minimising BIC

    Plus a diagnostic: MFP on the extended 15-power grid (fix #9) so we can
    tell whether SaDE's advantage is from the optimiser or the grid size.
    """

    # Extended power set used by SaDE. None = drop that power slot.
    POWER_SET = [None, -3, -2.5, -2, -1.5, -1, -0.5, -0.25,
                 0,    0.25, 0.5,  1,  1.5,  2,  2.5,  3]
    N_POWERS  = len(POWER_SET)  # 16

    def __init__(self, df, covariates, duration_col, event_col,
                 strata_cols=None):

        self.covariates   = covariates
        self.duration_col = duration_col
        self.event_col    = event_col
        self.strata_cols  = strata_cols or []

        self.df, self._scales = self._preprocess_positive(df, covariates)

        self._n_events = int(self.df[self.event_col].sum())
        self._n_total  = len(self.df)

        # Precompute all FP transforms on training data (before centring)
        self._precomp = {}
        self._log_x   = {}
        for col in covariates:
            x     = self.df[col].values.astype(float)
            log_x = np.log(x)
            self._log_x[col] = log_x
            for p in [p for p in self.POWER_SET if p is not None]:
                z = log_x if p == 0 else np.power(x, p)
                if np.isfinite(z).all():
                    self._precomp[(col, p)] = z

        # Training-sample feature means (for centring + test-time alignment, fix #4)
        self._train_means = {}   # keyed by final feature column name

        const_cols = {
            self.duration_col: self.df[self.duration_col].values,
            self.event_col:    self.df[self.event_col].values,
        }
        for c in self.strata_cols:
            const_cols[c] = self.df[c].values
        self._const_df = pd.DataFrame(const_cols, index=self.df.index)

        self.evaluation_cache  = {}
        self.best_val          = np.inf
        self.history: list     = []
        self.best_powers: list = []

        # Model objects
        self.km_model          = None
        self.km_strat_models   = {}
        self.traditional_model = None
        self.weibull_aft_model = None
        self.final_fp_model    = None
        self.mfp_model         = None    # standard 8-power MFP
        self.mfp_result        = None
        self.mfp_powers: list  = []
        self.mfp_ext_model     = None    # extended 15-power MFP (sensitivity)
        self.mfp_ext_result    = None

        # Intermediate DataFrames
        self._df_trad_final = None
        self._df_fp_final   = None
        self._df_mfp_final  = None
        self._df_mfp_ext_final = None

        self.metrics_: dict = {}

    # -----------------------------------------------------------------------
    # Preprocessing
    # -----------------------------------------------------------------------

    @staticmethod
    def _preprocess_positive(df, features, scales=None):
        df = df.copy()
        computed_scales = {}
        for col in features:
            x = df[col].astype(float)
            if (x <= 0).any():
                x = x - x.min() + 1e-5
            if scales is not None:
                scale = scales[col]
            else:
                mean_abs = np.mean(np.abs(x))
                scale = 1.0 if mean_abs == 0 else 10.0**np.floor(np.log10(mean_abs))
            computed_scales[col] = scale
            df[col] = x / scale
        return df, computed_scales

    # -----------------------------------------------------------------------
    # Canonical key — review fix #5 (key on powers, not indices)
    # -----------------------------------------------------------------------

    def _canonical_key(self, indices):
        """
        Rewrite of v11's keying: we now key on the decoded powers, so
        (None, p) and (p, None) map to the same cache entry.
        """
        pairs = []
        for i in range(0, len(indices), 2):
            p1 = self.POWER_SET[indices[i]]
            p2 = self.POWER_SET[indices[i+1]]
            if p1 is None and p2 is not None:
                p1, p2 = p2, None
            elif p1 is not None and p2 is not None and p1 > p2:
                p1, p2 = p2, p1
            pairs.append((p1, p2))
        return tuple(pairs)

    # -----------------------------------------------------------------------
    # FP feature generation — review fix #4 (centring)
    # -----------------------------------------------------------------------

    def _generate_fp_features(self, features, powers, store_means=False):
        """
        Build FP columns for training data. Each column is centred on its
        sample mean (fix #4). If store_means=True, the means are saved so
        that test-time prediction can use the same centring constants.
        """
        transformed = {}
        for col, (p1, p2) in zip(features, powers):
            active = sorted([p for p in (p1, p2) if p is not None],
                            key=lambda v: (v == 0, v))
            if not active:
                continue
            if len(active) == 1:
                p = active[0]
                arr = self._precomp.get((col, p))
                if arr is None: return None
                name = f'{col}_fp_{p}'
                m = float(arr.mean())
                transformed[name] = arr - m
                if store_means: self._train_means[name] = m
            else:
                pa, pb = active
                za = self._precomp.get((col, pa))
                if za is None: return None
                name1 = f'{col}_fp1_{pa}'
                m1 = float(za.mean())
                transformed[name1] = za - m1
                if store_means: self._train_means[name1] = m1
                if pa == pb:
                    zb_raw = za * self._log_x[col]
                    name2 = f'{col}_fp2_rep_{pb}'
                else:
                    zb_raw = self._precomp.get((col, pb))
                    if zb_raw is None: return None
                    name2 = f'{col}_fp2_{pb}'
                m2 = float(zb_raw.mean())
                transformed[name2] = zb_raw - m2
                if store_means: self._train_means[name2] = m2
        return transformed

    def _generate_fp_features_on(self, df, features, powers,
                                 training_means=None):
        """
        Build FP columns for an arbitrary DataFrame (e.g. simulation holdout
        or subsample). `training_means` maps feature name = mean of the
        corresponding training column; if provided, we subtract those means
        so the test predictions are aligned with the fitted model.
        If training_means is None, we compute fresh means from the input df
        (this is what we do when re-fitting on a subsample inside the
        simulation study).
        """
        transformed = {}
        for col, (p1, p2) in zip(features, powers):
            x     = df[col].values.astype(float)
            log_x = np.log(x)
            active = sorted([p for p in (p1, p2) if p is not None],
                            key=lambda v: (v == 0, v))
            if not active: continue

            def xp(p):
                z = log_x if p == 0 else np.power(x, p)
                return z if np.isfinite(z).all() else None

            if len(active) == 1:
                p = active[0]; arr = xp(p)
                if arr is None: return None
                name = f'{col}_fp_{p}'
                m = training_means[name] if training_means is not None \
                    and name in training_means else float(arr.mean())
                transformed[name] = arr - m
            else:
                pa, pb = active
                za = xp(pa)
                if za is None: return None
                name1 = f'{col}_fp1_{pa}'
                m1 = training_means[name1] if training_means is not None \
                    and name1 in training_means else float(za.mean())
                transformed[name1] = za - m1
                if pa == pb:
                    zb_raw = za * log_x
                    name2 = f'{col}_fp2_rep_{pb}'
                else:
                    zb_raw = xp(pb)
                    if zb_raw is None: return None
                    name2 = f'{col}_fp2_{pb}'
                m2 = training_means[name2] if training_means is not None \
                    and name2 in training_means else float(zb_raw.mean())
                transformed[name2] = zb_raw - m2
        return transformed

    # -----------------------------------------------------------------------
    # Objective function — BIC with n_total (review fix #1)
    # -----------------------------------------------------------------------

    def _objective_function(self, x):
        key = self._canonical_key(x)
        if key in self.evaluation_cache:
            return self.evaluation_cache[key]

        powers = list(key)
        fp_cols = self._generate_fp_features(self.covariates, powers)
        if not fp_cols:
            self.evaluation_cache[key] = 1e10
            return 1e10

        df_model = self._const_df.copy()
        for col_name, arr in fp_cols.items():
            df_model[col_name] = arr

        strata = self.strata_cols or None
        try:
            cph = CoxPHFitter(penalizer=0.0)
            cph.fit(df_model, duration_col=self.duration_col,
                    event_col=self.event_col, strata=strata,
                    show_progress=False)
            k   = len(cph.params_)
            # FIX #1: use n_total for BIC (matches mfp2 and Weibull convention)
            bic = -2*cph.log_likelihood_ + k*np.log(self._n_total)
            val = bic
        except Exception:
            val = 1e10

        self.evaluation_cache[key] = val
        if val < self.best_val: self.best_val = val
        return val

    def _random_init_pop(self, pop_size, rng):
        dim = 2 * len(self.covariates)
        return rng.integers(0, self.N_POWERS, size=(pop_size, dim))

    def _callback(self, best_x, best_f, gen):
        self.history.append(self.best_val)

    # -----------------------------------------------------------------------
    # IBS helper
    # -----------------------------------------------------------------------

    def _compute_ibs(self, cph, df_for_model):
        if not HAS_SKSURV: return None
        try:
            y_train = Surv.from_arrays(
                event=self.df[self.event_col].astype(bool).values,
                time =self.df[self.duration_col].values)
            t_min  = self.df[self.duration_col].min()
            t_max  = self.df[self.duration_col].max()
            times  = np.linspace(t_min, t_max*0.999, 100)
            surv = cph.predict_survival_function(df_for_model, times=times)
            return float(integrated_brier_score(y_train, y_train, surv.T.values, times))
        except Exception: return None

    def _compute_ibs_km(self):
        if not HAS_SKSURV or self.km_model is None: return None
        try:
            y = Surv.from_arrays(
                event=self.df[self.event_col].astype(bool).values,
                time =self.df[self.duration_col].values)
            t_min  = self.df[self.duration_col].min()
            t_max  = self.df[self.duration_col].max()
            times  = np.linspace(t_min, t_max*0.999, 100)
            km_sf  = self.km_model.survival_function_at_times(times).values
            n      = len(self.df)
            surv_matrix = np.tile(km_sf, (n, 1))
            return float(integrated_brier_score(y, y, surv_matrix, times))
        except Exception as e:
            print(f'  [!] KM IBS failed: {e}')
            return None

    # -----------------------------------------------------------------------
    # optimize — main entry point
    # -----------------------------------------------------------------------

    def _compute_pop_size(self, dim, pop_size=None, pop_multiplier=10,
                          pop_min=None, pop_max=None):
        """
        Data-adaptive population sizing for SaDE.

        Rule (grounded in EC literature):
          - Storn & Price 1997 recommend NP in [5*D, 10*D].
          - 10*D is the most widely applied default in SaDE/JADE/jDE papers.
          - Lower floor of max(20, 4*D) keeps mutation strategies feasible
            (DE/rand/2 and DE/curr-to-best/2 need >=6 distinct indices).
          - Upper cap of min(15*D, n_events) ties the search budget to
            dataset signal: the Cox partial-likelihood surface has limited
            effective resolution when events are sparse, so populations
            larger than the event count waste fitness evaluations.

        Parameters
        ----------
        dim : int
            Problem dimension (2 x n_covariates for FP2).
        pop_size : int or None
            If provided, overrides all computation and uses this literal NP.
        pop_multiplier : float
            Multiplier on dim for the baseline NP (default 10 = Storn-Price mid).
        pop_min, pop_max : int or None
            Override the auto-computed floor/cap.

        Returns
        -------
        NP   : int
        info : dict with keys 'target','floor','cap','final','override','multiplier'
        """
        if pop_min is None:
            pop_min = max(20, 4 * dim)
        if pop_max is None:
            pop_max = min(15 * dim, max(pop_min, self._n_events))
            pop_max = max(pop_max, pop_min)

        HARD_MIN = 6   # DE/rand/2 and DE/curr-to-best/2 need >=6 distinct

        if pop_size is not None:
            # Explicit override — bypass soft floor/cap, enforce only hard min
            target   = int(pop_size)
            override = True
            NP       = max(HARD_MIN, target)
        else:
            target   = int(round(pop_multiplier * dim))
            override = False
            NP       = int(np.clip(target, pop_min, pop_max))
        return NP, {
            'dim':        dim,
            'target':     target,
            'floor':      pop_min,
            'cap':        pop_max,
            'final':      NP,
            'override':   override,
            'multiplier': pop_multiplier if not override else None,
        }

    def optimize(self, maxiter=60, pop_size=None, pop_multiplier=10,
                 pop_min=None, pop_max=None, seed=42, max_evals=None):
        """
        Run SaDE with data-adaptive population sizing.

        Parameters
        ----------
        maxiter : int
            Maximum generations. Budget = NP * maxiter unless max_evals given.
        pop_size : int or None
            Absolute NP override. If given, pop_multiplier is ignored.
        pop_multiplier : float
            NP = pop_multiplier * dim, clipped to [pop_min, pop_max].
            Default 10 follows Storn & Price (1997) and is used in most
            SaDE/JADE/jDE implementations.
        pop_min, pop_max : int or None
            Manual floor and cap. Defaults:
              floor = max(20, 4 * dim)
              cap   = min(15 * dim, n_events)
            The event-count cap prevents wasted evaluations on small
            datasets — the BIC surface is effectively discretised by the
            event count.
        seed, max_evals : as before.
        """
        dim = 2 * len(self.covariates)

        # Data-adaptive population sizing
        NP, sz_info = self._compute_pop_size(
            dim, pop_size=pop_size, pop_multiplier=pop_multiplier,
            pop_min=pop_min, pop_max=pop_max)

        if max_evals is None:
            max_evals = NP * maxiter
        lp       = max(3, maxiter // 5)
        patience = max(3, maxiter // 2)

        print(f'Starting SaDE v12  (pure BIC, no penalizer, n_total convention)')
        print(f'  FP covariates : {self.covariates}')
        print(f'  Simultaneous  : {len(self.covariates)} covariates '
              f'| dim={dim}')
        if self.strata_cols:
            print(f'  Strata        : {self.strata_cols}')
        print(f'  Scales        : '
              + ', '.join(f'{c}/{s:.3g}' for c,s in self._scales.items()))
        print(f'  n={self._n_total}, n_events={self._n_events}')
        print(f'  Population sizing (data-adaptive, Storn-Price 1997 rule):')
        if sz_info['override']:
            print(f'    User override    : NP = {sz_info["target"]}')
        else:
            print(f'    Target ({sz_info["multiplier"]:g}*D)    : {sz_info["target"]}')
        print(f'    Floor            : {sz_info["floor"]}   '
              f'(= max(20, 4*D))')
        print(f'    Cap              : {sz_info["cap"]}   '
              f'(= min(15*D, n_events={self._n_events}))')
        print(f'    -> NP chosen     : {NP}')
        print(f'  MaxGens={maxiter} | Budget={max_evals} | Patience={patience}')

        rng      = np.random.default_rng(seed)
        init_pop = self._random_init_pop(NP, rng)
        pop_size = NP  # alias for downstream references
        engine   = _SaDE(
            func=self._objective_function,
            bounds=[(0, self.N_POWERS-1)]*dim,
            pop_size=pop_size, max_evals=max_evals,
            lp=lp, patience=patience, seed=seed, callback=self._callback)
        result = engine.run_opt(init_pop=init_pop)

        best_indices = result.x
        print('\n--- Optimal Power Selection ---')
        self.best_powers = []
        for i in range(len(self.covariates)):
            p1 = self.POWER_SET[best_indices[2*i]]
            p2 = self.POWER_SET[best_indices[2*i+1]]
            self.best_powers.append((p1, p2))
            active = [p for p in (p1, p2) if p is not None]
            fp_type = 'dropped' if not active else f'FP{len(active)}'
            print(f'  {self.covariates[i]:<22}: p1={str(p1):<7} p2={str(p2):<7}  [{fp_type}]')
        print(f'\n  Best BIC  : {result.fun:.4f}')
        print(f'  Gens      : {result.ngen}')
        print(f'  Evals     : {result.nfev} '
              f'(cache size: {len(self.evaluation_cache)})')

        names = ['DE/rand/1','DE/curr-to-best/2','DE/curr-to-rand/1','DE/rand/2']
        print('\n--- Strategy Usage ---')
        for i, name in enumerate(names):
            use = engine.strategy_counts[i]
            suc = engine.strategy_success[i]
            rate = 100*suc/use if use > 0 else 0.0
            print(f'  {name:<28}: used={use:4d}  success={suc:4d}  rate={rate:5.1f}%')

        self._plot_convergence()
        self._fit_final_models()

    def _plot_convergence(self):
        plt.figure(figsize=(10, 4))
        plt.plot(range(1, len(self.history)+1), self.history,
                 marker='o', ms=3, color='steelblue')
        plt.title('SaDE Convergence')
        plt.xlabel('Generation')
        plt.ylabel('BIC')
        plt.grid(True, alpha=0.4)
        plt.tight_layout()
        plt.show()

    # -----------------------------------------------------------------------
    # _fit_final_models — 5 models + extended-grid MFP sensitivity print
    # -----------------------------------------------------------------------

    def _fit_final_models(self):
        print('\n' + '='*72)
        print('FIVE-MODEL COMPARISON  (BIC uses n_total everywhere)')
        print('='*72)

        strata = self.strata_cols or None

        # Traditional covariate DataFrame
        seen, trad_cols = set(), []
        for c in (self.covariates + self.strata_cols +
                  [self.duration_col, self.event_col]):
            if c not in seen:
                trad_cols.append(c); seen.add(c)
        self._df_trad_final = self.df[trad_cols].copy()

        # == 1. Kaplan-Meier ==========================================
        print('\n[1/5] Fitting Kaplan-Meier...')
        self.km_model = KaplanMeierFitter()
        self.km_model.fit(
            durations  = self.df[self.duration_col],
            event_observed = self.df[self.event_col],
            label      = 'Kaplan-Meier (marginal)')
        self.km_strat_models = {}
        if self.strata_cols:
            strat_col = self.strata_cols[0]
            for val in sorted(self.df[strat_col].unique()):
                mask = self.df[strat_col] == val
                kmf  = KaplanMeierFitter()
                kmf.fit(
                    durations      = self.df.loc[mask, self.duration_col],
                    event_observed = self.df.loc[mask, self.event_col],
                    label          = f'KM {strat_col}={val}')
                self.km_strat_models[val] = kmf
        ibs_km = self._compute_ibs_km()
        median_km = self.km_model.median_survival_time_
        print(f'   Median survival time : {median_km:.4f}')
        print(f'   IBS (train)          : {ibs_km:.4f}' if ibs_km else '   IBS : N/A')

        # == 2. Traditional Cox PH ====================================
        print('\n[2/5] Fitting Traditional Cox PH...')
        self.traditional_model = CoxPHFitter(penalizer=0.0)
        self.traditional_model.fit(
            self._df_trad_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)
        k_t   = len(self.traditional_model.params_)
        bic_t = -2*self.traditional_model.log_likelihood_ + k_t*np.log(self._n_total)
        ci_t  = self.traditional_model.concordance_index_
        ibs_t = self._compute_ibs(self.traditional_model, self._df_trad_final)
        print(f'   C-index : {ci_t:.4f}   BIC : {bic_t:.2f}'
              + (f'   IBS : {ibs_t:.4f}' if ibs_t else ''))

        # == 3. Weibull AFT ===========================================
        print('\n[3/5] Fitting Weibull AFT...')
        df_aft = self.df[trad_cols].copy()
        self.weibull_aft_model = WeibullAFTFitter(penalizer=0.0)
        try:
            self.weibull_aft_model.fit(
                df_aft, duration_col=self.duration_col,
                event_col=self.event_col, show_progress=False)
            k_w   = self.weibull_aft_model.params_.shape[0]
            ll_w  = self.weibull_aft_model.log_likelihood_
            aic_w = self.weibull_aft_model.AIC_
            bic_w = -2*ll_w + k_w*np.log(self._n_total)
            ci_w  = self.weibull_aft_model.concordance_index_
            ibs_w = self._compute_ibs(self.weibull_aft_model, df_aft)
            print(f'   C-index : {ci_w:.4f}   AIC : {aic_w:.2f}   BIC : {bic_w:.2f}'
                  + (f'   IBS : {ibs_w:.4f}' if ibs_w else ''))
        except Exception as e:
            print(f'   [!] Weibull AFT failed: {e}')
            self.weibull_aft_model = None
            k_w = ci_w = aic_w = bic_w = ibs_w = None

        # == 4. MFP Cox (standard 8-power grid) =======================
        print('\n[4/5] Fitting MFP Cox (standard 8-power grid)...')
        mfp_std = MFPSelector(alpha_select=0.05, alpha_function=0.05,
                              max_cycles=10,
                              power_set=MFPSelector.STANDARD_POWERS)
        self.mfp_result = mfp_std.fit(
            self.df, self.covariates, self.duration_col, self.event_col,
            strata_cols=self.strata_cols)
        self.mfp_powers = [self.mfp_result['powers'][c] for c in self.covariates]
        mfp_feat = mfp_std.generate_fp_features(
            self.df, self.covariates, self.mfp_result)

        if mfp_feat:
            # Centre MFP features too (fix #4 applies equally)
            self._df_mfp_final = self._const_df.copy()
            for col_name, arr in mfp_feat.items():
                self._df_mfp_final[col_name] = arr - arr.mean()
            self.mfp_model = CoxPHFitter(penalizer=0.0)
            self.mfp_model.fit(
                self._df_mfp_final,
                duration_col=self.duration_col, event_col=self.event_col,
                strata=strata, show_progress=False)
            k_mfp   = len(self.mfp_model.params_)
            bic_mfp = -2*self.mfp_model.log_likelihood_ + k_mfp*np.log(self._n_total)
            ci_mfp  = self.mfp_model.concordance_index_
            ibs_mfp = self._compute_ibs(self.mfp_model, self._df_mfp_final)
            print(f'   C-index : {ci_mfp:.4f}   BIC : {bic_mfp:.2f}'
                  + (f'   IBS : {ibs_mfp:.4f}' if ibs_mfp else ''))
        else:
            print('   [!] MFP selected no variables — using null model.')
            k_mfp = ci_mfp = bic_mfp = ibs_mfp = None

        # == 5. FP Cox (SaDE) =========================================
        print('\n[5/5] Fitting FP Cox (SaDE)...')
        fp_cols = self._generate_fp_features(self.covariates, self.best_powers,
                                             store_means=True)
        if fp_cols is None: fp_cols = {}
        self._df_fp_final = self._const_df.copy()
        for col_name, arr in fp_cols.items():
            self._df_fp_final[col_name] = arr

        self.final_fp_model = CoxPHFitter(penalizer=0.0)
        self.final_fp_model.fit(
            self._df_fp_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)
        k_fp   = len(self.final_fp_model.params_)
        bic_fp = -2*self.final_fp_model.log_likelihood_ + k_fp*np.log(self._n_total)
        ci_fp  = self.final_fp_model.concordance_index_
        ibs_fp = self._compute_ibs(self.final_fp_model, self._df_fp_final)
        print(f'   C-index : {ci_fp:.4f}   BIC : {bic_fp:.2f}'
              + (f'   IBS : {ibs_fp:.4f}' if ibs_fp else ''))

        self.metrics_ = {
            'Kaplan-Meier': {
                'C-index': 'N/A',   'BIC': 'N/A', 'AIC': 'N/A',
                'IBS': round(ibs_km, 4) if ibs_km is not None else 'N/A',
                'k': 'N/A', 'median_T': round(median_km, 4)},
            'Cox PH (trad)': {
                'C-index': round(ci_t, 4), 'BIC': round(bic_t, 2),
                'AIC': 'N/A (partial)',
                'IBS': round(ibs_t, 4) if ibs_t is not None else 'N/A',
                'k': k_t},
            'Weibull AFT': {
                'C-index': round(ci_w, 4) if ci_w is not None else 'N/A',
                'BIC': round(bic_w, 2) if bic_w is not None else 'N/A',
                'AIC': round(aic_w, 2) if aic_w is not None else 'N/A',
                'IBS': round(ibs_w, 4) if ibs_w is not None else 'N/A',
                'k': k_w},
            'MFP Cox (trad FP)': {
                'C-index': round(ci_mfp, 4) if ci_mfp is not None else 'N/A',
                'BIC': round(bic_mfp, 2) if bic_mfp is not None else 'N/A',
                'AIC': 'N/A (partial)',
                'IBS': round(ibs_mfp, 4) if ibs_mfp is not None else 'N/A',
                'k': k_mfp},
            'FP Cox (SaDE)': {
                'C-index': round(ci_fp, 4), 'BIC': round(bic_fp, 2),
                'AIC': 'N/A (partial)',
                'IBS': round(ibs_fp, 4) if ibs_fp is not None else 'N/A',
                'k': k_fp},
        }

        self._print_comparison_table()

        # == Sensitivity: MFP on the extended power grid (fix #9) =====
        self._fit_extended_mfp_sensitivity()

        self._print_model_equations()
        self._plot_survival_curves(df_aft)

        self._test_ph_assumption(
            self.traditional_model, self._df_trad_final, 'Traditional Cox PH')
        if self.mfp_model is not None:
            self._test_ph_assumption(
                self.mfp_model, self._df_mfp_final, 'MFP Cox (trad FP)')
        self._test_ph_assumption(
            self.final_fp_model, self._df_fp_final, 'FP Cox (SaDE)')

    # -----------------------------------------------------------------------
    # Extended-grid MFP sensitivity check (review fix #9)
    # -----------------------------------------------------------------------

    def _fit_extended_mfp_sensitivity(self):
        print('\n' + '='*72)
        print('SENSITIVITY CHECK — MFP on the extended 15-power grid')
        print('='*72)
        print('Fair-comparison run: MFP uses the same real powers as SaDE (minus None).')
        print('If SaDE still has the lower BIC, the win is from the optimiser/objective,')
        print('not from the grid size.')

        extended_powers = [p for p in self.POWER_SET if p is not None]
        mfp_ext = MFPSelector(alpha_select=0.05, alpha_function=0.05,
                              max_cycles=10, power_set=extended_powers)
        try:
            self.mfp_ext_result = mfp_ext.fit(
                self.df, self.covariates, self.duration_col, self.event_col,
                strata_cols=self.strata_cols)
            mfp_ext_feat = mfp_ext.generate_fp_features(
                self.df, self.covariates, self.mfp_ext_result)
            if mfp_ext_feat:
                self._df_mfp_ext_final = self._const_df.copy()
                for col_name, arr in mfp_ext_feat.items():
                    self._df_mfp_ext_final[col_name] = arr - arr.mean()
                strata = self.strata_cols or None
                self.mfp_ext_model = CoxPHFitter(penalizer=0.0)
                self.mfp_ext_model.fit(
                    self._df_mfp_ext_final,
                    duration_col=self.duration_col, event_col=self.event_col,
                    strata=strata, show_progress=False)
                k_ext   = len(self.mfp_ext_model.params_)
                bic_ext = -2*self.mfp_ext_model.log_likelihood_ + k_ext*np.log(self._n_total)
                ci_ext  = self.mfp_ext_model.concordance_index_
                ibs_ext = self._compute_ibs(self.mfp_ext_model, self._df_mfp_ext_final)

                fp_bic = self.metrics_['FP Cox (SaDE)']['BIC']
                fp_ci  = self.metrics_['FP Cox (SaDE)']['C-index']
                try:
                    bic_delta = float(fp_bic) - bic_ext
                except:
                    bic_delta = None

                print(f'\n  MFP-extended : C-index={ci_ext:.4f}   BIC={bic_ext:.2f}'
                      + (f'   IBS={ibs_ext:.4f}' if ibs_ext else '')
                      + f'   k={k_ext}')
                print(f'  FP SaDE      : C-index={fp_ci}   BIC={fp_bic}')
                if bic_delta is not None:
                    if bic_delta < -2:
                        print(f'  = SaDE wins BIC by {-bic_delta:.2f} — real optimiser advantage')
                    elif bic_delta > 2:
                        print(f'  = MFP-extended wins BIC by {bic_delta:.2f} — SaDE under-explores')
                    else:
                        print(f'  = BIC tie (|ΔBIC|={abs(bic_delta):.2f}) — both optimisers found the same region')
            else:
                print('  [!] MFP-extended selected no variables.')
        except Exception as e:
            print(f'  [!] MFP-extended failed: {e}')

    # -----------------------------------------------------------------------
    # Comparison table
    # -----------------------------------------------------------------------

    def _print_comparison_table(self):
        bar  = '='*72
        sep  = '-'*72
        W    = 18
        models = list(self.metrics_.keys())
        print(f'\n{bar}')
        print('FIVE-MODEL SUMMARY TABLE')
        print(bar)
        hdr = f"{'Metric':<16}"
        for m in models:
            hdr += f' | {m:>{W}}'
        print(hdr); print(sep)
        for metric in ['C-index', 'BIC', 'AIC', 'IBS', 'k']:
            row = f'{metric:<16}'
            for m in models:
                val = self.metrics_[m].get(metric, 'N/A')
                row += f' | {str(val):>{W}}'
            print(row)
        print(sep)
        print('Best (↑ C-index, ↓ BIC, ↓ IBS):')
        for metric, higher_better in [('C-index', True), ('BIC', False), ('IBS', False)]:
            vals = {}
            for m in models:
                v = self.metrics_[m].get(metric, 'N/A')
                try: vals[m] = float(v)
                except: pass
            if vals:
                best = max(vals, key=vals.get) if higher_better \
                       else min(vals, key=vals.get)
                direction = '↑' if higher_better else '↓'
                print(f'  {metric:<10}: {best}  ({direction} {vals[best]:.4f})')

        def _delta_bic(name_a, name_b):
            try:
                bic_a = float(self.metrics_[name_a]['BIC'])
                bic_b = float(self.metrics_[name_b]['BIC'])
                delta = bic_a - bic_b
                winner = name_b if delta > 0 else name_a
                print(f'\n  ΔBIC ({name_a} − {name_b}) = {delta:.2f}  '
                      f'({winner} preferred)')
                if abs(delta) > 10:   print('   = Very strong evidence')
                elif abs(delta) > 6:  print('   = Strong evidence')
                elif abs(delta) > 2:  print('   = Positive evidence')
                else:                 print('   = Models essentially equivalent')
            except Exception: pass
        _delta_bic('Cox PH (trad)', 'FP Cox (SaDE)')
        _delta_bic('Cox PH (trad)', 'MFP Cox (trad FP)')
        _delta_bic('MFP Cox (trad FP)', 'FP Cox (SaDE)')
        print(bar)
        print('Note: all BICs use n_total in the penalty (mfp2 convention).')

    # -----------------------------------------------------------------------
    # Survival curves and equations
    # -----------------------------------------------------------------------

    def _plot_survival_curves(self, df_aft):
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        t_min  = self.df[self.duration_col].min()
        t_max  = self.df[self.duration_col].max()
        times  = np.linspace(t_min, t_max, 200)

        ax = axes[0]
        km_sf = self.km_model.survival_function_at_times(times)
        ax.plot(times, km_sf, color='gray', lw=2.5, ls='--',
                label='Kaplan-Meier (marginal)')
        strat_colors = plt.cm.Greys(
            np.linspace(0.35, 0.75, len(self.km_strat_models)))
        for (val, kmf), sc in zip(self.km_strat_models.items(), strat_colors):
            sf = kmf.survival_function_at_times(times)
            ax.plot(times, sf, color=sc, lw=1.2, ls=':', label=kmf.label)

        mean_profile_trad = self._df_trad_final[
            [c for c in self._df_trad_final.columns
             if c not in [self.duration_col, self.event_col]]].mean()
        try:
            sf_cox = self.traditional_model.predict_survival_function(
                mean_profile_trad.to_frame().T, times=times).squeeze()
            ax.plot(times, sf_cox, color='steelblue', lw=2.5,
                    label='Cox PH (traditional, mean profile)')
        except Exception as e: print(f'  [!] Cox PH curve failed: {e}')

        if self.weibull_aft_model is not None:
            try:
                mean_profile_aft = df_aft[
                    [c for c in df_aft.columns
                     if c not in [self.duration_col, self.event_col]]].mean()
                sf_aft = self.weibull_aft_model.predict_survival_function(
                    mean_profile_aft.to_frame().T, times=times).squeeze()
                ax.plot(times, sf_aft, color='darkorange', lw=2.5, ls='-.',
                        label='Weibull AFT (mean profile)')
            except Exception as e: print(f'  [!] Weibull AFT curve failed: {e}')

        if self.mfp_model is not None and self._df_mfp_final is not None:
            try:
                mean_profile_mfp = self._df_mfp_final[
                    [c for c in self._df_mfp_final.columns
                     if c not in [self.duration_col, self.event_col]]].mean()
                sf_mfp = self.mfp_model.predict_survival_function(
                    mean_profile_mfp.to_frame().T, times=times).squeeze()
                ax.plot(times, sf_mfp, color='seagreen', lw=2.5, ls='-.',
                        label='MFP Cox (trad FP, mean profile)')
            except Exception as e: print(f'  [!] MFP Cox curve failed: {e}')

        try:
            mean_profile_fp = self._df_fp_final[
                [c for c in self._df_fp_final.columns
                 if c not in [self.duration_col, self.event_col]]].mean()
            sf_fp = self.final_fp_model.predict_survival_function(
                mean_profile_fp.to_frame().T, times=times).squeeze()
            ax.plot(times, sf_fp, color='crimson', lw=2.5,
                    label='FP Cox (SaDE, mean profile)')
        except Exception as e: print(f'  [!] FP Cox curve failed: {e}')

        ax.set_xlabel('Time'); ax.set_ylabel('S(t)')
        ax.set_title('Survival Curve Comparison — All 5 Models')
        ax.set_ylim(0, 1.02); ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

        ax2 = axes[1]
        self.km_model.plot_survival_function(ax=ax2, ci_show=True, color='gray')
        for val, kmf in self.km_strat_models.items():
            kmf.plot_survival_function(ax=ax2, ci_show=False)
        ibs_km  = self.metrics_['Kaplan-Meier']['IBS']
        ibs_cox = self.metrics_['Cox PH (trad)']['IBS']
        ibs_aft = self.metrics_['Weibull AFT']['IBS']
        ibs_mfp = self.metrics_['MFP Cox (trad FP)']['IBS']
        ibs_fp  = self.metrics_['FP Cox (SaDE)']['IBS']
        anno = (f"IBS  KM={ibs_km}  Cox={ibs_cox}\n"
                f"     AFT={ibs_aft}  MFP={ibs_mfp}  FP={ibs_fp}")
        ax2.text(0.02, 0.08, anno, transform=ax2.transAxes,
                 fontsize=8, va='bottom',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        ax2.set_xlabel('Time'); ax2.set_ylabel('S(t)')
        ax2.set_title('KM Curve(s) with 95% CI'); ax2.grid(True, alpha=0.3)

        plt.suptitle('Five-Model Survival Function Comparison',
                     fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

    def _print_model_equations(self):
        bar = '='*72
        print(f'\n{bar}')
        print('MODEL EQUATIONS')
        print(bar)

        def _fmt_p(p):
            if p is None:  return 'None'
            if p == 0:     return 'x^0 = ln(x)'
            if p == 0.5:   return 'x^0.5 = √x'
            if p == 1:     return 'x^1 (linear)'
            if p == 2:     return 'x^2 (quadratic)'
            return f'x^{p}'

        print('\n== 1. KAPLAN-MEIER  (non-parametric, no covariates) ==')
        print('  S(t) = Π_{tᵢ ≤ t} (1 − dᵢ/nᵢ)')
        if self.km_model is not None:
            print(f'  Median survival time : {self.km_model.median_survival_time_:.4f}')

        print('\n== 2. TRADITIONAL COX PH  (linear covariates) ==')
        print('  log[ h(t|x) / h₀(t) ] =')
        if self.traditional_model is not None:
            for feat, coef in self.traditional_model.params_.items():
                sign = '+' if coef >= 0 else '−'
                print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={len(self.traditional_model.params_)},  '
                  f'C-index={self.traditional_model.concordance_index_:.4f}')

        print('\n== 3. WEIBULL AFT  (fully parametric, log-time) ==')
        print('  log T = μ(x) + σ·ε,  ε ~ Gumbel')
        if self.weibull_aft_model is not None:
            params = self.weibull_aft_model.params_
            try:
                for (sub, feat), coef in params.items():
                    if sub == 'lambda_':
                        sign = '+' if coef >= 0 else '−'
                        print(f'    {sign} {abs(coef):.6f} × {feat}')
                for (sub, feat), coef in params.items():
                    if sub == 'rho_':
                        print(f'    σ = exp({coef:.6f}) = {np.exp(coef):.6f}')
            except Exception:
                for feat, coef in params.items():
                    sign = '+' if coef >= 0 else '−'
                    print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={self.weibull_aft_model.params_.shape[0]},  '
                  f'C-index={self.weibull_aft_model.concordance_index_:.4f}')

        print('\n== 4. MFP COX (Traditional FP, 8 powers) ==')
        print('  log[ h(t|x) / h₀(t) ] =')
        if self.mfp_model is not None:
            for feat, coef in self.mfp_model.params_.items():
                sign = '+' if coef >= 0 else '−'
                print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={len(self.mfp_model.params_)},  '
                  f'C-index={self.mfp_model.concordance_index_:.4f}')
            print('\n  MFP Power Selection (FSP/LRT):')
            for cov in self.covariates:
                ft = self.mfp_result['fp_types'][cov]
                pw = self.mfp_result['powers'][cov]
                print(f'    {cov:<22}: {ft:<8}  powers={pw}')

        print('\n== 5. FP COX (SaDE, 15 powers + None) ==')
        print('  log[ h(t|x) / h₀(t) ] =')
        if self.final_fp_model is not None:
            for feat, coef in self.final_fp_model.params_.items():
                sign = '+' if coef >= 0 else '−'
                print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={len(self.final_fp_model.params_)},  '
                  f'C-index={self.final_fp_model.concordance_index_:.4f}')

        if self.best_powers:
            print('\n  FP Power Annotation:')
            for cov, (p1, p2) in zip(self.covariates, self.best_powers):
                active  = [p for p in (p1, p2) if p is not None]
                fp_type = 'dropped' if not active else f'FP{len(active)}'
                scale   = self._scales.get(cov, 1.0)
                print(f'    {cov}  [{fp_type}]  (scale ÷ {scale:.4g})')
                for idx, p in enumerate(active, 1):
                    print(f'      term {idx}: {_fmt_p(p)}')
                if len(active) == 2 and active[0] == active[1]:
                    print(f'      [repeated power] term 2 = x^{active[0]} · ln(x)')
        print(bar)

    def _test_ph_assumption(self, model, df_model, model_name='Model',
                            p_threshold=0.05):
        bar = '='*64
        print(f'\n{bar}')
        print(f'  PH ASSUMPTION TEST — {model_name}')
        print(bar)
        print(f'  H₀: log-HR constant over time  |  α = {p_threshold}')

        print('\n  [A] lifelines check_assumptions():')
        try:
            model.check_assumptions(df_model, p_value_threshold=p_threshold,
                                    show_plots=True, advice=False)
        except Exception as e:
            print(f'  [!] Failed: {e}')

        print(f'\n  [B] Pearson ρ(Schoenfeld residual, ranked event time):')
        print(f'  {"-"*60}')
        any_flagged = False
        try:
            schoenfeld  = model.compute_residuals(df_model, kind='schoenfeld')
            event_mask  = df_model[self.event_col].astype(bool).values
            event_times = df_model[self.duration_col].values[event_mask]
            ranked_t    = pd.Series(event_times).rank().values
            for col in schoenfeld.columns:
                res = schoenfeld[col].values
                if len(res) != len(ranked_t): continue
                rho, pval = scipy_stats.pearsonr(res, ranked_t)
                flag = '  ⚠ |ρ|>0.2' if abs(rho) > 0.2 else ''
                if flag: any_flagged = True
                print(f'  {col:<38} ρ={rho:+.4f}  p={pval:.4f}{flag}')
            print()
            if any_flagged:
                print('  ⚠  Consider stratifying or using time-varying coefficients.')
            else:
                print('  ✓  No strong evidence of PH violation.')
        except Exception as e:
            print(f'  [!] Schoenfeld failed: {e}')
        print(bar)

    # -----------------------------------------------------------------------
    # Simulation study — review fix #10 (adds out-of-sample C-index)
    # -----------------------------------------------------------------------

    def run_simulation_study(self, n_sims=1000, sample_frac=0.90,
                             seed=0, show_progress_every=100):
        """
        Bootstrap-style simulation: each iteration draws a stratified
        90% subsample, refits all four covariate models using the FIXED
        powers from optimize(), and records metrics on BOTH the training
        90% and the 10% holdout — including holdout C-index (fix #10).
        """
        if self._df_trad_final is None or self._df_fp_final is None:
            raise RuntimeError("Call optimize() before run_simulation_study().")

        bar = "=" * 72
        print(f"\n{bar}")
        print(f"SIMULATION STUDY  (n_sims={n_sims}, sample_frac={sample_frac:.0%})")
        print(bar)
        print(f"  FP/MFP powers FIXED from prior optimize() call.")
        print(f"  Models: Cox PH, Weibull AFT, MFP Cox, FP Cox")
        print(f"  Metrics per iteration: C-index (train + TEST), BIC, IBS (train + test)")
        print(f"  Each iteration: {int(len(self.df)*sample_frac)} / {len(self.df)} rows "
              f"(stratified by event status)\n")

        rng    = np.random.default_rng(seed)
        strata = self.strata_cols or None

        seen, trad_cols = set(), []
        for c in (self.covariates + self.strata_cols +
                  [self.duration_col, self.event_col]):
            if c not in seen:
                trad_cols.append(c); seen.add(c)
        aft_cols = list(trad_cols)

        metrics_rows = []
        n_failed = 0

        for sim_i in range(n_sims):
            events_idx   = self.df.index[self.df[self.event_col].astype(bool)]
            censored_idx = self.df.index[~self.df[self.event_col].astype(bool)]
            n_ev  = max(1, int(round(len(events_idx)   * sample_frac)))
            n_cen = max(1, int(round(len(censored_idx) * sample_frac)))
            sampled_ev  = rng.choice(events_idx,   size=n_ev,  replace=False)
            sampled_cen = rng.choice(censored_idx, size=n_cen, replace=False)
            idx_sample  = np.concatenate([sampled_ev, sampled_cen])

            df_sim = self.df.loc[idx_sample].copy()
            holdout_idx = self.df.index.difference(pd.Index(idx_sample))
            df_hold = self.df.loc[holdout_idx].copy() if len(holdout_idx) > 0 else None

            df_sim_trad = df_sim[trad_cols].copy()
            df_sim_aft  = df_sim[aft_cols].copy()

            # FP features on subsample — recompute means on the subsample
            fp_feat = self._generate_fp_features_on(
                df_sim, self.covariates, self.best_powers)
            if fp_feat is None:
                n_failed += 1; continue

            # Capture the subsample means so the holdout uses the same centring
            sub_means = {}
            for col, (p1, p2) in zip(self.covariates, self.best_powers):
                x     = df_sim[col].values.astype(float)
                log_x = np.log(x)
                active = sorted([p for p in (p1, p2) if p is not None],
                                key=lambda v: (v == 0, v))
                if not active: continue
                def xp(p): return log_x if p == 0 else np.power(x, p)
                if len(active) == 1:
                    p = active[0]
                    sub_means[f'{col}_fp_{p}'] = float(xp(p).mean())
                else:
                    pa, pb = active
                    sub_means[f'{col}_fp1_{pa}'] = float(xp(pa).mean())
                    if pa == pb:
                        sub_means[f'{col}_fp2_rep_{pb}'] = float((xp(pa)*log_x).mean())
                    else:
                        sub_means[f'{col}_fp2_{pb}'] = float(xp(pb).mean())

            const_cols_sim = {
                self.duration_col: df_sim[self.duration_col].values,
                self.event_col:    df_sim[self.event_col].values,
            }
            for c in self.strata_cols:
                const_cols_sim[c] = df_sim[c].values
            df_sim_fp = pd.DataFrame(const_cols_sim, index=df_sim.index)
            for cn, arr in fp_feat.items(): df_sim_fp[cn] = arr

            # MFP features on subsample using the fixed powers selected on the full data
            mfp_sel_obj = MFPSelector(power_set=MFPSelector.STANDARD_POWERS)
            mfp_feat_sim = mfp_sel_obj.generate_fp_features(
                df_sim, self.covariates, self.mfp_result)
            df_sim_mfp = None
            sub_means_mfp = {}
            if mfp_feat_sim:
                df_sim_mfp = pd.DataFrame(const_cols_sim, index=df_sim.index)
                for cn, arr in mfp_feat_sim.items():
                    m = float(arr.mean())
                    df_sim_mfp[cn] = arr - m
                    sub_means_mfp[cn] = m

            # Build holdout dataframes using subsample-derived centring means
            has_holdout  = False
            df_hold_trad = df_hold_aft = df_hold_fp = df_hold_mfp = None
            y_holdout = None
            if df_hold is not None and int(df_hold[self.event_col].sum()) >= 2:
                try:
                    df_hold_trad = df_hold[trad_cols].copy()
                    df_hold_aft  = df_hold[aft_cols].copy()

                    fp_feat_hold = self._generate_fp_features_on(
                        df_hold, self.covariates, self.best_powers,
                        training_means=sub_means)
                    if fp_feat_hold is not None:
                        _const_h = {
                            self.duration_col: df_hold[self.duration_col].values,
                            self.event_col:    df_hold[self.event_col].values,
                        }
                        for c in self.strata_cols:
                            _const_h[c] = df_hold[c].values
                        df_hold_fp = pd.DataFrame(_const_h, index=df_hold.index)
                        for cn, arr in fp_feat_hold.items(): df_hold_fp[cn] = arr

                    # MFP holdout features
                    mfp_feat_hold = mfp_sel_obj.generate_fp_features(
                        df_hold, self.covariates, self.mfp_result)
                    if mfp_feat_hold:
                        df_hold_mfp = pd.DataFrame(_const_h, index=df_hold.index)
                        for cn, arr in mfp_feat_hold.items():
                            df_hold_mfp[cn] = arr - sub_means_mfp.get(cn, arr.mean())

                    if HAS_SKSURV:
                        y_holdout = Surv.from_arrays(
                            event=df_hold[self.event_col].astype(bool).values,
                            time =df_hold[self.duration_col].values)
                    has_holdout = True
                except Exception:
                    has_holdout = False

            n_total_sim = len(df_sim)
            row_metrics = {}

            # == 1. Cox PH =============================================
            try:
                cph = CoxPHFitter(penalizer=0.0)
                cph.fit(df_sim_trad, duration_col=self.duration_col,
                        event_col=self.event_col, strata=strata,
                        show_progress=False)
                k_t   = len(cph.params_)
                bic_t = -2*cph.log_likelihood_ + k_t*np.log(n_total_sim)
                row_metrics["cox_cindex_train"] = cph.concordance_index_
                row_metrics["cox_bic"]          = bic_t

                # Holdout C-index (fix #10)
                if has_holdout:
                    try:
                        haz = cph.predict_partial_hazard(df_hold_trad)
                        row_metrics["cox_cindex_test"] = concordance_index(
                            df_hold[self.duration_col].values,
                            -np.asarray(haz).ravel(),
                            df_hold[self.event_col].values)
                    except Exception: pass

                if HAS_SKSURV:
                    try:
                        y_tr = Surv.from_arrays(
                            event=df_sim[self.event_col].astype(bool).values,
                            time =df_sim[self.duration_col].values)
                        t_lo = df_sim[self.duration_col].min()
                        t_hi = df_sim[self.duration_col].max()*0.999
                        if t_lo < t_hi:
                            times = np.linspace(t_lo, t_hi, 80)
                            sf = cph.predict_survival_function(df_sim_trad, times=times)
                            row_metrics["cox_ibs_train"] = float(integrated_brier_score(
                                y_tr, y_tr, sf.T.values, times))
                        if has_holdout and y_holdout is not None:
                            t_lo_h = df_hold[self.duration_col].min()
                            t_hi_h = df_hold[self.duration_col].max()*0.999
                            if t_lo_h < t_hi_h:
                                times_h = np.linspace(t_lo_h, t_hi_h, 80)
                                sf_h = cph.predict_survival_function(df_hold_trad, times=times_h)
                                row_metrics["cox_ibs_test"] = float(integrated_brier_score(
                                    y_tr, y_holdout, sf_h.T.values, times_h))
                    except Exception: pass
            except Exception: pass

            # == 2. Weibull AFT ========================================
            try:
                wft = WeibullAFTFitter(penalizer=0.0)
                wft.fit(df_sim_aft, duration_col=self.duration_col,
                        event_col=self.event_col, show_progress=False)
                k_w   = wft.params_.shape[0]
                bic_w = -2*wft.log_likelihood_ + k_w*np.log(n_total_sim)
                row_metrics["aft_cindex_train"] = wft.concordance_index_
                row_metrics["aft_bic"]          = bic_w

                if has_holdout:
                    try:
                        pred_t = wft.predict_median(df_hold_aft)
                        # AFT median is higher=longer survival; concordance_index wants
                        # higher=longer survival (positive sign).
                        row_metrics["aft_cindex_test"] = concordance_index(
                            df_hold[self.duration_col].values,
                            np.asarray(pred_t).ravel(),
                            df_hold[self.event_col].values)
                    except Exception: pass

                if HAS_SKSURV:
                    try:
                        y_tr = Surv.from_arrays(
                            event=df_sim[self.event_col].astype(bool).values,
                            time =df_sim[self.duration_col].values)
                        t_lo = df_sim[self.duration_col].min()
                        t_hi = df_sim[self.duration_col].max()*0.999
                        if t_lo < t_hi:
                            times = np.linspace(t_lo, t_hi, 80)
                            sf = wft.predict_survival_function(df_sim_aft, times=times)
                            row_metrics["aft_ibs_train"] = float(integrated_brier_score(
                                y_tr, y_tr, sf.T.values, times))
                        if has_holdout and y_holdout is not None:
                            t_lo_h = df_hold[self.duration_col].min()
                            t_hi_h = df_hold[self.duration_col].max()*0.999
                            if t_lo_h < t_hi_h:
                                times_h = np.linspace(t_lo_h, t_hi_h, 80)
                                sf_h = wft.predict_survival_function(df_hold_aft, times=times_h)
                                row_metrics["aft_ibs_test"] = float(integrated_brier_score(
                                    y_tr, y_holdout, sf_h.T.values, times_h))
                    except Exception: pass
            except Exception: pass

            # == 3. MFP Cox ============================================
            if df_sim_mfp is not None:
                try:
                    cph_mfp = CoxPHFitter(penalizer=0.0)
                    cph_mfp.fit(df_sim_mfp, duration_col=self.duration_col,
                                event_col=self.event_col, strata=strata,
                                show_progress=False)
                    k_m   = len(cph_mfp.params_)
                    bic_m = -2*cph_mfp.log_likelihood_ + k_m*np.log(n_total_sim)
                    row_metrics["mfp_cindex_train"] = cph_mfp.concordance_index_
                    row_metrics["mfp_bic"]          = bic_m

                    if has_holdout and df_hold_mfp is not None:
                        try:
                            haz = cph_mfp.predict_partial_hazard(df_hold_mfp)
                            row_metrics["mfp_cindex_test"] = concordance_index(
                                df_hold[self.duration_col].values,
                                -np.asarray(haz).ravel(),
                                df_hold[self.event_col].values)
                        except Exception: pass

                    if HAS_SKSURV:
                        try:
                            y_tr = Surv.from_arrays(
                                event=df_sim[self.event_col].astype(bool).values,
                                time =df_sim[self.duration_col].values)
                            t_lo = df_sim[self.duration_col].min()
                            t_hi = df_sim[self.duration_col].max()*0.999
                            if t_lo < t_hi:
                                times = np.linspace(t_lo, t_hi, 80)
                                sf = cph_mfp.predict_survival_function(df_sim_mfp, times=times)
                                row_metrics["mfp_ibs_train"] = float(integrated_brier_score(
                                    y_tr, y_tr, sf.T.values, times))
                            if has_holdout and y_holdout is not None and df_hold_mfp is not None:
                                t_lo_h = df_hold[self.duration_col].min()
                                t_hi_h = df_hold[self.duration_col].max()*0.999
                                if t_lo_h < t_hi_h:
                                    times_h = np.linspace(t_lo_h, t_hi_h, 80)
                                    sf_h = cph_mfp.predict_survival_function(df_hold_mfp, times=times_h)
                                    row_metrics["mfp_ibs_test"] = float(integrated_brier_score(
                                        y_tr, y_holdout, sf_h.T.values, times_h))
                        except Exception: pass
                except Exception: pass

            # == 4. FP Cox (SaDE) ======================================
            try:
                cph_fp = CoxPHFitter(penalizer=0.0)
                cph_fp.fit(df_sim_fp, duration_col=self.duration_col,
                           event_col=self.event_col, strata=strata,
                           show_progress=False)
                k_f   = len(cph_fp.params_)
                bic_f = -2*cph_fp.log_likelihood_ + k_f*np.log(n_total_sim)
                row_metrics["fp_cindex_train"] = cph_fp.concordance_index_
                row_metrics["fp_bic"]          = bic_f

                if has_holdout and df_hold_fp is not None:
                    try:
                        haz = cph_fp.predict_partial_hazard(df_hold_fp)
                        row_metrics["fp_cindex_test"] = concordance_index(
                            df_hold[self.duration_col].values,
                            -np.asarray(haz).ravel(),
                            df_hold[self.event_col].values)
                    except Exception: pass

                if HAS_SKSURV:
                    try:
                        y_tr = Surv.from_arrays(
                            event=df_sim[self.event_col].astype(bool).values,
                            time =df_sim[self.duration_col].values)
                        t_lo = df_sim[self.duration_col].min()
                        t_hi = df_sim[self.duration_col].max()*0.999
                        if t_lo < t_hi:
                            times = np.linspace(t_lo, t_hi, 80)
                            sf = cph_fp.predict_survival_function(df_sim_fp, times=times)
                            row_metrics["fp_ibs_train"] = float(integrated_brier_score(
                                y_tr, y_tr, sf.T.values, times))
                        if has_holdout and y_holdout is not None and df_hold_fp is not None:
                            t_lo_h = df_hold[self.duration_col].min()
                            t_hi_h = df_hold[self.duration_col].max()*0.999
                            if t_lo_h < t_hi_h:
                                times_h = np.linspace(t_lo_h, t_hi_h, 80)
                                sf_h = cph_fp.predict_survival_function(df_hold_fp, times=times_h)
                                row_metrics["fp_ibs_test"] = float(integrated_brier_score(
                                    y_tr, y_holdout, sf_h.T.values, times_h))
                    except Exception: pass
            except Exception: pass

            metrics_rows.append(row_metrics)
            if (sim_i + 1) % show_progress_every == 0:
                print(f"  sim {sim_i+1:4d}/{n_sims}  "
                      f"(failed so far: {n_failed})")

        metrics_df = pd.DataFrame(metrics_rows)

        # == Summary ====================================================
        print(f"\n  Completed {len(metrics_df)} simulations  (failed: {n_failed})")
        summary_rows = []
        for col in metrics_df.columns:
            v = metrics_df[col].dropna()
            if len(v) == 0: continue
            summary_rows.append({
                'metric': col,
                'mean':   v.mean(),
                'std':    v.std(),
                'cv%':    100*v.std()/abs(v.mean()) if v.mean() != 0 else np.nan,
                'q2.5':   v.quantile(0.025),
                'median': v.median(),
                'q97.5':  v.quantile(0.975),
            })
        summary = pd.DataFrame(summary_rows)
        print('\n' + '='*72)
        print('SIMULATION SUMMARY')
        print('='*72)
        print(summary.to_string(index=False, float_format='%.4f'))

        # == Plot: distributions of out-of-sample C-index ===============
        test_c_cols = [c for c in metrics_df.columns if c.endswith('_cindex_test')]
        if test_c_cols:
            fig, ax = plt.subplots(figsize=(10, 5))
            data = [metrics_df[c].dropna().values for c in test_c_cols]
            labels = [c.replace('_cindex_test', '').upper() for c in test_c_cols]
            bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5)
            colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
            for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                patch.set_facecolor(color); patch.set_alpha(0.6)
            ax.set_ylabel('Out-of-sample C-index (10% holdout)')
            ax.set_title(f'Test-set C-index distribution over {len(metrics_df)} simulations')
            ax.grid(True, alpha=0.3, axis='y')
            plt.tight_layout(); plt.show()

        return {
            'metrics_df': metrics_df,
            'summary':    summary,
            'n_sims':     len(metrics_df),
            'n_failed':   n_failed,
            'sample_frac': sample_frac,
        }


print('FPCoxOptimizer v12 loaded (all review fixes applied).')


## Example Dataset Runs

Each `optimize()` call runs SaDE, fits the five comparison models, prints the BIC / C-index / IBS table, and runs an extended-grid MFP sensitivity check.

With the new defaults (`popsize=20, maxiter=60`), the 10-covariate PBC run takes ~2–4 minutes on a typical machine.


In [ ]:
# == Dataset 1: Simulated ====================================================
sd_df = pd.read_csv('../../../data/preprocess-data/preprocess_simulated_data.csv')

optimizer_sd = FPCoxOptimizer(
    df           = sd_df,
    covariates   = ['Age'],
    duration_col = 'Time',
    event_col    = 'Event',
    strata_cols  = ['Treatment', 'Sex'],
)
optimizer_sd.optimize(maxiter=60, seed=42)   # adaptive NP (Storn-Price 10*D, capped by n_events)

print('\nFP Cox model summary (Simulated):')
optimizer_sd.final_fp_model.print_summary()


In [ ]:
# == Dataset 2: PBC (Primary Biliary Cirrhosis — Mayo Clinic) =============
pbc_df = pd.read_csv('../../../data/preprocess-data/preprocess_pbc.csv',
                     index_col=0)

PBC_COVARIATES = ['age', 'albumin', 'protime', 'stage',
                  'log_bili', 'log_chol', 'log_copper',
                  'log_alk_phos', 'log_ast', 'log_trig']
PBC_STRATA     = ['edema']

optimizer_pbc = FPCoxOptimizer(
    df           = pbc_df,
    covariates   = PBC_COVARIATES,
    duration_col = 'time',
    event_col    = 'status_binary',
    strata_cols  = PBC_STRATA,
)
optimizer_pbc.optimize(maxiter=60, seed=42)   # adaptive NP (Storn-Price 10*D, capped by n_events)

print('\nFP Cox model summary (PBC):')
optimizer_pbc.final_fp_model.print_summary()


In [ ]:
# == Dataset 3: GBSG =========================================================
gb_df = pd.read_csv('../../../data/preprocess-data/preprocess_gbsg.csv')

GB_COVARIATES = ['log_pgr', 'log_nodes', 'log_er', 'age', 'size']
GB_STRATA     = ['meno', 'grade', 'hormon']

optimizer_gb = FPCoxOptimizer(
    df           = gb_df,
    covariates   = GB_COVARIATES,
    duration_col = 'rfstime',
    event_col    = 'status',
    strata_cols  = GB_STRATA,
)
optimizer_gb.optimize(maxiter=60, seed=42)   # adaptive NP (Storn-Price 10*D, capped by n_events)

print('\nFP Cox model summary (GBSG):')
optimizer_gb.final_fp_model.print_summary()


## Simulation Study

Monte-Carlo subsample simulation: 1,000 draws of 90% of each dataset (stratified by event status). For each draw, all four covariate models (Cox PH, Weibull AFT, MFP Cox, FP Cox) are refit with their selected powers held fixed, and **out-of-sample** C-index + IBS are measured on the 10% holdout. This is the headline out-of-sample comparison for the paper.

Expect 10–30 minutes per dataset at `n_sims=1000`. Drop to `n_sims=200` for quick iteration while developing.


In [ ]:
# == Simulation: Simulated dataset =======================================
sim_results_sd = optimizer_sd.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)


In [ ]:
# == Simulation: PBC =====================================================
sim_results_pbc = optimizer_pbc.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)


In [ ]:
# == Simulation: GBSG ====================================================
sim_results_gb = optimizer_gb.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)
